In [2]:
from pathlib import Path
import xml.etree.ElementTree as ET

def print_entries_summary(file_name: str, max_entries: int | None = 5) -> None:
    """
    Print a simple summary of the first `max_entries` trace entries.
    Set max_entries=None to print all (could be very long).
    """
    notebook_dir = Path.cwd()
    traces_dir = notebook_dir.parent / "traces"
    file_path = traces_dir / file_name

    try:
        tree = ET.parse(file_path)
        root = tree.getroot()

        entries = root.find("entries")
        if entries is None:
            print("No <entries> section found.")
            return

        print("Game Entries")
        print("-" * 40)

        count = 0
        for entry in entries.findall("rts.TraceEntry"):
            time = entry.get("time", "?")
            print(f"Entry time = {time}")

            # --- Physical game state ---
            pgs = entry.find("rts.PhysicalGameState")
            if pgs is not None:
                w = pgs.get("width", "?")
                h = pgs.get("height", "?")
                print(f"  Map: {w} x {h}")

                # Players
                players_node = pgs.find("players")
                if players_node is not None:
                    print("  Players:")
                    for p in players_node.findall("rts.Player"):
                        pid = p.get("ID", "?")
                        res = p.get("resources", "?")
                        print(f"    Player {pid}: resources={res}")

                # Units
                units_node = pgs.find("units")
                if units_node is not None:
                    print("  Units:")
                    for u in units_node.findall("rts.units.Unit"):
                        utype = u.get("type", "?")
                        uid = u.get("ID", "?")
                        player = u.get("player", "?")
                        x = u.get("x", "?")
                        y = u.get("y", "?")
                        hp = u.get("hitpoints", "?")
                        res = u.get("resources", "?")
                        print(f"    ID={uid}, type={utype}, player={player}, "
                              f"pos=({x},{y}), hp={hp}, resources={res}")

            # --- Actions ---
            actions_node = entry.find("actions")
            if actions_node is not None:
                print("  Actions:")
                any_action = False
                for a in actions_node.findall("action"):
                    any_action = True
                    uid = a.get("unitID", "?")
                    ua = a.find("UnitAction")
                    if ua is not None:
                        atype = ua.get("type", "?")
                        param = ua.get("parameter")
                        ux = ua.get("x")
                        uy = ua.get("y")
                        ut = ua.get("unitType")
                        extra = []
                        if param is not None:
                            extra.append(f"parameter={param}")
                        if ux is not None and uy is not None:
                            extra.append(f"target=({ux},{uy})")
                        if ut is not None:
                            extra.append(f"unitType={ut}")
                        extra_str = ", ".join(extra) if extra else ""
                        print(f"    unitID={uid}, type={atype}"
                              + (f", {extra_str}" if extra_str else ""))
                if not any_action:
                    print("    (no actions)")
            print("-" * 40)

            count += 1
            if max_entries is not None and count >= max_entries:
                break

    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except ET.ParseError as e:
        print(f"XML parse error: {e}")
    except OSError as e:
        print(f"Error reading file: {e}")

# Example call:
print_entries_summary("AggrobotAStarPathFindingVsRandomBiasedAI-0-0.xml", max_entries=5)


Game Entries
----------------------------------------
Entry time = 0
  Map: 8 x 8
  Players:
    Player 0: resources=5
    Player 1: resources=5
  Units:
    ID=0, type=Resource, player=-1, pos=(0,0), hp=1, resources=20
    ID=1, type=Resource, player=-1, pos=(7,7), hp=1, resources=20
    ID=2, type=Base, player=0, pos=(2,1), hp=10, resources=0
    ID=3, type=Base, player=1, pos=(5,6), hp=10, resources=0
  Actions:
    (no actions)
----------------------------------------
Entry time = 0
  Map: 8 x 8
  Players:
    Player 0: resources=5
    Player 1: resources=5
  Units:
    ID=0, type=Resource, player=-1, pos=(0,0), hp=1, resources=20
    ID=1, type=Resource, player=-1, pos=(7,7), hp=1, resources=20
    ID=2, type=Base, player=0, pos=(2,1), hp=10, resources=0
    ID=3, type=Base, player=1, pos=(5,6), hp=10, resources=0
  Actions:
    unitID=2, type=4, parameter=0, unitType=Worker
    unitID=3, type=4, parameter=1, unitType=Worker
----------------------------------------
Entry time = 50

In [13]:
from pathlib import Path
import xml.etree.ElementTree as ET

def print_unit_type_table(file_name: str) -> None:
    notebook_dir = Path.cwd()
    traces_dir = notebook_dir.parent / "traces"
    file_path = traces_dir / file_name

    try:
        tree = ET.parse(file_path)
        root = tree.getroot()

        # Find the UnitTypeTable node
        utt = root.find(".//rts.units.UnitTypeTable")

        if utt is None:
            print("No <rts.units.UnitTypeTable> found in XML.")
            return

        print("Unit type Table")
        print("-" * 40)

        # Iterate over all UnitType elements inside the table
        for ut in utt.findall("rts.units.UnitType"):
            attrs = ut.attrib  # dict of attributes
            name = attrs.get("name", "<no-name>")
            print(f"UnitType: {name}")
            for k, v in attrs.items():
                if k == "name":
                    continue
                print(f"  {k}: {v}")
            print()  # blank line between unit types

    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except ET.ParseError as e:
        print(f"XML parse error: {e}")
    except OSError as e:
        print(f"Error reading file: {e}")

# Call with your file
print_unit_type_table("AggrobotAStarPathFindingVsRandomBiasedAI-0-0.xml")


Unit type Table
----------------------------------------
UnitType: Resource
  ID: 0
  cost: 1
  hp: 1
  minDamage: 1
  maxDamage: 1
  attackRange: 1
  produceTime: 10
  moveTime: 10
  attackTime: 10
  harvestTime: 10
  returnTime: 10
  harvestAmount: 1
  sightRadius: 0
  isResource: true
  isStockpile: false
  canHarvest: false
  canMove: false
  canAttack: false

UnitType: Base
  ID: 1
  cost: 10
  hp: 10
  minDamage: 1
  maxDamage: 1
  attackRange: 1
  produceTime: 250
  moveTime: 10
  attackTime: 10
  harvestTime: 10
  returnTime: 10
  harvestAmount: 1
  sightRadius: 5
  isResource: false
  isStockpile: true
  canHarvest: false
  canMove: false
  canAttack: false

UnitType: Barracks
  ID: 2
  cost: 5
  hp: 4
  minDamage: 1
  maxDamage: 1
  attackRange: 1
  produceTime: 200
  moveTime: 10
  attackTime: 10
  harvestTime: 10
  returnTime: 10
  harvestAmount: 1
  sightRadius: 3
  isResource: false
  isStockpile: false
  canHarvest: false
  canMove: false
  canAttack: false

UnitType: Wo

In [2]:
from pathlib import Path
import math
import xml.etree.ElementTree as ET

# ========= GLOBAL TRACE DIRECTORY =========
TRACE_DIR = "traces/12x12"   # change this once to switch directory

def get_trace_path(file_name: str) -> Path:
    notebook_dir = Path.cwd()
    traces_dir = notebook_dir.parent / TRACE_DIR
    return traces_dir / file_name

# ========= GLOBAL OUTPUT BUFFER =========
log_lines = []

def log(msg: str) -> None:
    log_lines.append(msg)

# ========= HELPERS =========

def build_unit_cost_table(root) -> dict:
    utt = root.find(".//rts.units.UnitTypeTable")
    cost_table = {}
    if utt is not None:
        for ut in utt.findall("rts.units.UnitType"):
            name = ut.get("name")
            cost = ut.get("cost")
            if name is not None and cost is not None:
                cost_table[name] = int(cost)
    return cost_table

def count_units_built(root, player_id: str) -> dict:
    entries = root.find("entries")
    counts = {}
    if entries is None:
        return counts

    seen_ids = set()
    for entry in entries.findall("rts.TraceEntry"):
        pgs = entry.find("rts.PhysicalGameState")
        if pgs is None:
            continue
        units = pgs.find("units")
        if units is None:
            continue

        for u in units.findall("rts.units.Unit"):
            if u.get("player") != player_id:
                continue
            uid = u.get("ID")
            if uid is None or uid in seen_ids:
                continue
            seen_ids.add(uid)
            utype = u.get("type")
            if utype is None:
                continue
            counts[utype] = counts.get(utype, 0) + 1
    return counts

def get_player_resources(entry, player_id: str) -> int:
    pgs = entry.find("rts.PhysicalGameState")
    if pgs is None:
        return 0
    players = pgs.find("players")
    if players is None:
        return 0
    for p in players.findall("rts.Player"):
        if p.get("ID") == player_id:
            return int(p.get("resources", "0"))
    return 0

def total_map_resources(root) -> int:
    entries = root.find("entries")
    if entries is None:
        return 0
    first_entry = next(iter(entries.findall("rts.TraceEntry")), None)
    if first_entry is None:
        return 0

    pgs = first_entry.find("rts.PhysicalGameState")
    if pgs is None:
        return 0
    units = pgs.find("units")
    if units is None:
        return 0

    total = 0
    for u in units.findall("rts.units.Unit"):
        if u.get("type") == "Resource" and u.get("player") == "-1":
            total += int(u.get("resources", "0"))
    return total

# ========= MAIN ANALYSIS FUNCTIONS =========

def first_barracks_time(file_name: str, player_id: str) -> None:
    file_path = get_trace_path(file_name)
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return

        first_time = None
        for entry in entries.findall("rts.TraceEntry"):
            t = entry.get("time")
            pgs = entry.find("rts.PhysicalGameState")
            if pgs is None:
                continue
            units = pgs.find("units")
            if units is None:
                continue

            for u in units.findall("rts.units.Unit"):
                if u.get("type") == "Barracks" and u.get("player") == player_id:
                    first_time = t
                    break
            if first_time is not None:
                break

        if first_time is None:
            log(f"Player {player_id} never has a Barracks in this trace.")
        else:
            log(f"Player {player_id} first has a Barracks at trace time = {first_time}")

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")

def barracks_count(file_name: str, player_id: str) -> None:
    file_path = get_trace_path(file_name)
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return

        barracks_ids = set()
        for entry in entries.findall("rts.TraceEntry"):
            pgs = entry.find("rts.PhysicalGameState")
            if pgs is None:
                continue
            units = pgs.find("units")
            if units is None:
                continue

            for u in units.findall("rts.units.Unit"):
                if u.get("type") == "Barracks" and u.get("player") == player_id:
                    uid = u.get("ID")
                    if uid is not None:
                        barracks_ids.add(uid)

        log(f"Player {player_id} built {len(barracks_ids)} Barracks in this trace.")

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")

def harvested_resources(file_name: str, player_id: str) -> None:
    file_path = get_trace_path(file_name)
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()

        cost_table = build_unit_cost_table(root)
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return

        all_entries = entries.findall("rts.TraceEntry")
        if not all_entries:
            log("Trace has no entries.")
            return

        first_entry = all_entries[0]
        last_entry = all_entries[-1]

        r_initial = get_player_resources(first_entry, player_id)
        r_final = get_player_resources(last_entry, player_id)

        built_counts = count_units_built(root, player_id)

        total_spent = 0
        for unit_type, n in built_counts.items():
            cost = cost_table.get(unit_type, 0)
            total_spent += cost * n

        harvested = (r_final - r_initial) + total_spent

        log(f"Player {player_id} approximately harvested {harvested} resources in this game.")
        log(f"  (initial={r_initial}, final={r_final}, spent={total_spent})")

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")

def harvested_resources_percent(file_name: str, player_id: str) -> None:
    file_path = get_trace_path(file_name)
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()

        cost_table = build_unit_cost_table(root)
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return

        all_entries = entries.findall("rts.TraceEntry")
        if not all_entries:
            log("Trace has no entries.")
            return

        first_entry = all_entries[0]
        last_entry = all_entries[-1]

        r_initial = get_player_resources(first_entry, player_id)
        r_final = get_player_resources(last_entry, player_id)

        built_counts = count_units_built(root, player_id)
        total_spent = 0
        for unit_type, n in built_counts.items():
            cost = cost_table.get(unit_type, 0)
            total_spent += cost * n

        harvested = (r_final - r_initial) + total_spent
        total_initial_resources = total_map_resources(root)

        if total_initial_resources > 0:
            percent = 100.0 * harvested / total_initial_resources
        else:
            percent = 0.0

        log(f"Total initial resources on map: {total_initial_resources}")
        log(f"Player {player_id} approx harvested: {harvested}")
        log(f"Player {player_id} harvested ~{percent:.2f}% of all resources on the map.")

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")

def workers_built(file_name: str, player_id: str) -> None:
    file_path = get_trace_path(file_name)
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return

        seen_ids = set()
        workers_built_count = 0

        for entry in entries.findall("rts.TraceEntry"):
            pgs = entry.find("rts.PhysicalGameState")
            if pgs is None:
                continue
            units = pgs.find("units")
            if units is None:
                continue

            for u in units.findall("rts.units.Unit"):
                if u.get("player") != player_id:
                    continue
                if u.get("type") != "Worker":
                    continue
                uid = u.get("ID")
                if uid is None or uid in seen_ids:
                    continue
                seen_ids.add(uid)
                workers_built_count += 1

        log(f"Player {player_id} built {workers_built_count} Workers in this trace.")

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")

def workers_built_milestones(file_name: str, player_id: str) -> None:
    file_path = get_trace_path(file_name)
    milestones = [100, 200, 400, 800]
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return

        milestone_counts = {m: None for m in milestones}
        seen_ids = set()
        workers_built_count = 0

        for entry in entries.findall("rts.TraceEntry"):
            t_str = entry.get("time")
            if t_str is None:
                continue
            t = int(t_str)

            pgs = entry.find("rts.PhysicalGameState")
            if pgs is None:
                continue
            units = pgs.find("units")
            if units is None:
                continue

            for u in units.findall("rts.units.Unit"):
                if u.get("player") != player_id:
                    continue
                if u.get("type") != "Worker":
                    continue
                uid = u.get("ID")
                if uid is None or uid in seen_ids:
                    continue
                seen_ids.add(uid)
                workers_built_count += 1

            for m in milestones:
                if milestone_counts[m] is None and t >= m:
                    milestone_counts[m] = workers_built_count

        log(f"Workers built by player {player_id} at milestones (cumulative):")
        for m in milestones:
            if milestone_counts[m] is not None:
                log(f"  time >= {m}: {milestone_counts[m]} Workers")
            else:
                log(f"  time >= {m}: (milestone not reached in this trace)")

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")

def game_length_ticks(file_name: str) -> None:
    file_path = get_trace_path(file_name)
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return

        all_entries = entries.findall("rts.TraceEntry")
        if not all_entries:
            log("Trace has no entries.")
            return

        last_entry = all_entries[-1]
        t_str = last_entry.get("time", "0")
        try:
            t = int(t_str)
        except ValueError:
            t = 0

        log(f"Game length: {t} ticks")

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")

def player_won(file_name: str, player_id: str, enemy_id: str) -> None:
    file_path = get_trace_path(file_name)
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return

        all_entries = entries.findall("rts.TraceEntry")
        if not all_entries:
            log("Trace has no entries.")
            return

        last_entry = all_entries[-1]
        pgs = last_entry.find("rts.PhysicalGameState")
        if pgs is None:
            log("No PhysicalGameState in last entry.")
            return

        units = pgs.find("units")
        if units is None:
            log("No units section in last entry.")
            return

        has_me = False
        has_enemy = False
        for u in units.findall("rts.units.Unit"):
            p = u.get("player")
            if p == player_id:
                has_me = True
            elif p == enemy_id:
                has_enemy = True

        if has_me and not has_enemy:
            log(f"Player {player_id}: WIN")
        elif has_enemy and not has_me:
            log(f"Player {player_id}: LOSS")
        else:
            log(f"Player {player_id}: DRAW/UNDECIDED (both or neither have units)")

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")

def combat_units_built(file_name: str, player_id: str) -> None:
    file_path = get_trace_path(file_name)
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return

        seen_ids = set()
        light = heavy = ranged = 0

        for entry in entries.findall("rts.TraceEntry"):
            pgs = entry.find("rts.PhysicalGameState")
            if pgs is None:
                continue
            units = pgs.find("units")
            if units is None:
                continue

            for u in units.findall("rts.units.Unit"):
                if u.get("player") != player_id:
                    continue
                utype = u.get("type")
                if utype not in ("Light", "Heavy", "Ranged"):
                    continue
                uid = u.get("ID")
                if uid is None or uid in seen_ids:
                    continue
                seen_ids.add(uid)
                if utype == "Light":
                    light += 1
                elif utype == "Heavy":
                    heavy += 1
                elif utype == "Ranged":
                    ranged += 1

        log(f"Player {player_id} built in total:")
        log(f"  Light units : {light}")
        log(f"  Heavy units : {heavy}")
        log(f"  Ranged units: {ranged}")

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")

def first_attack_time(file_name: str, player_id: str) -> None:
    file_path = get_trace_path(file_name)
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return

        first_time = None
        for entry in entries.findall("rts.TraceEntry"):
            t = entry.get("time")
            if t is None:
                continue

            pgs = entry.find("rts.PhysicalGameState")
            if pgs is None:
                continue
            units_node = pgs.find("units")
            if units_node is None:
                continue

            unit_player = {}
            for u in units_node.findall("rts.units.Unit"):
                uid = u.get("ID")
                p = u.get("player")
                if uid is not None:
                    unit_player[uid] = p

            actions_node = entry.find("actions")
            if actions_node is None:
                continue

            for a in actions_node.findall("action"):
                uid = a.get("unitID")
                if uid is None:
                    continue
                if unit_player.get(uid) != player_id:
                    continue
                ua = a.find("UnitAction")
                if ua is None:
                    continue
                if ua.get("type") == "5":
                    first_time = t
                    break
            if first_time is not None:
                break

        if first_time is None:
            log(f"Player {player_id} never issues an ATTACK action in this trace.")
        else:
            log(f"Player {player_id} first ATTACK action at trace time = {first_time}")

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")

def total_attacks(file_name: str, player_id: str) -> int:
    file_path = get_trace_path(file_name)
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return 0

        attack_count = 0
        for entry in entries.findall("rts.TraceEntry"):
            pgs = entry.find("rts.PhysicalGameState")
            if pgs is None:
                continue
            units_node = pgs.find("units")
            if units_node is None:
                continue

            unit_player = {}
            for u in units_node.findall("rts.units.Unit"):
                uid = u.get("ID")
                p = u.get("player")
                if uid is not None:
                    unit_player[uid] = p

            actions_node = entry.find("actions")
            if actions_node is None:
                continue

            for a in actions_node.findall("action"):
                uid = a.get("unitID")
                if uid is None:
                    continue
                if unit_player.get(uid) != player_id:
                    continue
                ua = a.find("UnitAction")
                if ua is None:
                    continue
                if ua.get("type") == "5":
                    attack_count += 1

        log(f"Player {player_id} issued {attack_count} ATTACK actions in this trace.")
        return attack_count

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")
    return 0

def attacks_per_trace(file_name: str, player_id: str) -> None:
    file_path = get_trace_path(file_name)
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return

        all_entries = entries.findall("rts.TraceEntry")
        num_entries = len(all_entries)
        if num_entries == 0:
            log("Trace has no entries.")
            return

        total_attacks_count = 0
        for entry in all_entries:
            pgs = entry.find("rts.PhysicalGameState")
            if pgs is None:
                continue
            units_node = pgs.find("units")
            if units_node is None:
                continue

            unit_player = {}
            for u in units_node.findall("rts.units.Unit"):
                uid = u.get("ID")
                p = u.get("player")
                if uid is not None:
                    unit_player[uid] = p

            actions_node = entry.find("actions")
            if actions_node is None:
                continue

            for a in actions_node.findall("action"):
                uid = a.get("unitID")
                if uid is None:
                    continue
                if unit_player.get(uid) != player_id:
                    continue
                ua = a.find("UnitAction")
                if ua is None:
                    continue
                if ua.get("type") == "5":
                    total_attacks_count += 1

        attacks_per_entry = total_attacks_count / num_entries
        log(f"Player {player_id} issued {total_attacks_count} ATTACK actions in total.")
        log(f"Number of trace entries: {num_entries}")
        log(f"Average ATTACKs per trace entry: {attacks_per_entry:.4f}")

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")

def max_attacks_in_single_trace(file_name: str, player_id: str) -> None:
    file_path = get_trace_path(file_name)
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return

        max_attacks = 0
        max_time = None

        for entry in entries.findall("rts.TraceEntry"):
            t = entry.get("time")
            if t is None:
                continue

            pgs = entry.find("rts.PhysicalGameState")
            if pgs is None:
                continue
            units_node = pgs.find("units")
            if units_node is None:
                continue

            unit_player = {}
            for u in units_node.findall("rts.units.Unit"):
                uid = u.get("ID")
                p = u.get("player")
                if uid is not None:
                    unit_player[uid] = p

            actions_node = entry.find("actions")
            if actions_node is None:
                continue

            attacks_this_entry = 0
            for a in actions_node.findall("action"):
                uid = a.get("unitID")
                if uid is None:
                    continue
                if unit_player.get(uid) != player_id:
                    continue
                ua = a.find("UnitAction")
                if ua is None:
                    continue
                if ua.get("type") == "5":
                    attacks_this_entry += 1

            if attacks_this_entry > max_attacks:
                max_attacks = attacks_this_entry
                max_time = t

        if max_time is None:
            log(f"Player {player_id} never issues an ATTACK action in this trace.")
        else:
            log(f"Max ATTACKs by player {player_id} in a single trace entry: {max_attacks}")
            log(f"Occurred at trace time = {max_time}")

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")

def avg_distance_to_enemy_base(file_name: str, player_id: str, enemy_id: str) -> None:
    file_path = get_trace_path(file_name)
    milestones = [100, 200, 400, 800]
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return

        all_entries = entries.findall("rts.TraceEntry")
        if not all_entries:
            log("Trace has no entries.")
            return

        enemy_base_pos = None
        for entry in all_entries:
            pgs = entry.find("rts.PhysicalGameState")
            if pgs is None:
                continue
            units = pgs.find("units")
            if units is None:
                continue
            for u in units.findall("rts.units.Unit"):
                if u.get("type") == "Base" and u.get("player") == enemy_id:
                    x = u.get("x")
                    y = u.get("y")
                    if x is not None and y is not None:
                        enemy_base_pos = (int(x), int(y))
                        break
            if enemy_base_pos is not None:
                break

        if enemy_base_pos is None:
            log(f"Could not find enemy base (player {enemy_id} Base) in trace.")
            return

        ex, ey = enemy_base_pos
        results = {m: (None, None) for m in milestones}

        for m in milestones:
            entry_for_m = None
            for entry in all_entries:
                t_str = entry.get("time")
                if t_str is None:
                    continue
                t = int(t_str)
                if t >= m:
                    entry_for_m = (t, entry)
                    break
            if entry_for_m is None:
                continue

            t, entry = entry_for_m
            pgs = entry.find("rts.PhysicalGameState")
            if pgs is None:
                continue
            units = pgs.find("units")
            if units is None:
                continue

            distances = []
            for u in units.findall("rts.units.Unit"):
                if u.get("player") != player_id:
                    continue
                ux = u.get("x")
                uy = u.get("y")
                if ux is None or uy is None:
                    continue
                ux = int(ux)
                uy = int(uy)
                d = math.sqrt((ux - ex) ** 2 + (uy - ey) ** 2)
                distances.append(d)

            if distances:
                avg_d = sum(distances) / len(distances)
                results[m] = (avg_d, t)
            else:
                results[m] = (None, t)

        log(f"Average distance of player-{player_id} units to enemy base (player {enemy_id} Base):")
        log(f"Enemy base at: {enemy_base_pos}")
        for m in milestones:
            avg_d, t = results[m]
            if t is None:
                log(f"  time >= {m}: milestone not reached in this trace")
            elif avg_d is None:
                log(f"  time >= {m} (first time={t}): no player-{player_id} units present")
            else:
                log(f"  time >= {m} (first time={t}): avg distance = {avg_d:.3f}")

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")

def combat_units_milestones(file_name: str, player_id: str) -> None:
    file_path = get_trace_path(file_name)
    milestones = [100, 200, 400, 800]
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        entries = root.find("entries")
        if entries is None:
            log("No <entries> section found.")
            return

        all_entries = entries.findall("rts.TraceEntry")
        if not all_entries:
            log("Trace has no entries.")
            return

        results = {}
        for m in milestones:
            chosen = None
            for entry in all_entries:
                t_str = entry.get("time")
                if t_str is None:
                    continue
                t = int(t_str)
                if t >= m:
                    chosen = (t, entry)
                    break
            if chosen is None:
                results[m] = None
                continue

            t, entry = chosen
            pgs = entry.find("rts.PhysicalGameState")
            if pgs is None:
                results[m] = (t, 0, 0, 0)
                continue
            units = pgs.find("units")
            if units is None:
                results[m] = (t, 0, 0, 0)
                continue

            light = heavy = ranged = 0
            for u in units.findall("rts.units.Unit"):
                if u.get("player") != player_id:
                    continue
                utype = u.get("type")
                if utype == "Light":
                    light += 1
                elif utype == "Heavy":
                    heavy += 1
                elif utype == "Ranged":
                    ranged += 1

            results[m] = (t, light, heavy, ranged)

        log(f"Player {player_id} combat units at milestones:")
        for m in milestones:
            val = results[m]
            if val is None:
                log(f"  time >= {m}: milestone not reached in this trace")
            else:
                t, light, heavy, ranged = val
                log(f"  time >= {m} (first time={t}): Light={light}, Heavy={heavy}, Ranged={ranged}")

    except FileNotFoundError:
        log(f"File not found: {file_path}")
    except ET.ParseError as e:
        log(f"XML parse error: {e}")
    except OSError as e:
        log(f"Error reading file: {e}")

# ========= MASTER CALLER =========

def analyze_trace(file_name: str, player_id: str, enemy_id: str = "1") -> Path:
    global log_lines
    log_lines = []

    log(f"=== Analyzing trace: {file_name}, player {player_id} vs {enemy_id} ===")
    first_barracks_time(file_name, player_id)
    barracks_count(file_name, player_id)
    combat_units_milestones(file_name, player_id)
    avg_distance_to_enemy_base(file_name, player_id, enemy_id)
    max_attacks_in_single_trace(file_name, player_id)
    attacks_per_trace(file_name, player_id)
    first_attack_time(file_name, player_id)
    combat_units_built(file_name, player_id)
    game_length_ticks(file_name)
    player_won(file_name, player_id, enemy_id)
    workers_built_milestones(file_name, player_id)
    workers_built(file_name, player_id)
    harvested_resources(file_name, player_id)
    harvested_resources_percent(file_name, player_id)

    # use just the directory name + generic suffix
    dir_name = Path(TRACE_DIR).name
    out_name = f"analyzed_traces_{dir_name}_ALL.txt"
    out_path = Path.cwd() / out_name

    with out_path.open("a", encoding="utf-8") as f:
        for line in log_lines:
            f.write(line + "\n")
        f.write("\n")

    return out_path

def infer_players_from_name(file_name: str):
    """
    If name starts with 'tiamat' (case-insensitive): player_id = '1', enemy_id = '0'
    If name starts with 'mayari'                     : player_id = '0', enemy_id = '1'
    Else default: player_id = '0', enemy_id = '1'
    """
    lower = file_name.lower()
    if lower.startswith("tiamat"):
        return "1", "0"
    elif lower.startswith("mayari"):
        return "0", "1"
    else:
        return "0", "1"

# ========= DRIVER: ANALYZE ALL TRACES IN DIRECTORY =========

def analyze_all_traces_in_dir():
    traces_dir = Path.cwd().parent / TRACE_DIR
    dir_name = Path(TRACE_DIR).name
    out_name = f"analyzed_traces_{dir_name}_ALL.txt"
    out_path = Path.cwd() / out_name

    # start from a clean file
    if out_path.exists():
        out_path.unlink()

    # analyze every .xml trace in the directory
    for path in sorted(traces_dir.glob("*.xml")):
        file_name = path.name
        player_id, enemy_id = infer_players_from_name(file_name)
        analyze_trace(file_name, player_id=player_id, enemy_id=enemy_id)

    print(f"Finished analyzing all traces in {traces_dir}")
    print(f"Results written to {out_path}")

# Run this to process all traces in TRACE_DIR
analyze_all_traces_in_dir()


Finished analyzing all traces in C:\Users\bikep\IdeaProjects\myMicroRTSBot\traces\12x12
Results written to C:\Users\bikep\IdeaProjects\myMicroRTSBot\pythonCode\analyzed_traces_12x12_ALL.txt


In [3]:
import re
from pathlib import Path

log_path = Path("analyzed_traces_12x12_ALL.txt")  # adjust if needed

with log_path.open("r", encoding="utf-8") as f:
    full_text = f.read().strip()

blocks = [b.strip() for b in full_text.split("\n\n") if b.strip()]

COLS = [
    "FirstBarracks",
    "NumBarracks",
    "Light100",
    "Heavy100",
    "Ranged100",
    "Light200",
    "Heavy200",
    "Ranged200",
    "Light400",
    "Heavy400",
    "Ranged400",
    "Light800",
    "Heavy800",
    "Ranged800",
    "AvgDist100",
    "AvgDist200",
    "AvgDist400",
    "AvgDist800",
    "MaxAttacksSingleTrace",
    "TimeMaxAttacks",
    "TotalAttacks",
    "AvgAttacksPerTrace",
    "FirstAttackTime",
    "TotalLight",
    "TotalHeavy",
    "TotalRanged",
    "GameLength",
    "PlayerWin",
    "Workers100",
    "Workers200",
    "Workers400",
    "Workers800",
    "TotalWorkers",
    "PercentMapHarvested",
]

MILESTONES = [100, 200, 400, 800]

def carry_forward_milestones(data: dict) -> None:
    # Workers: carry forward, then fall back to TotalWorkers
    last = None
    for t in MILESTONES:
        key = f"Workers{t}"
        v = data.get(key, -1)
        if v != -1:
            last = v
        else:
            if last is not None:
                data[key] = last
    total_workers = data.get("TotalWorkers", -1)
    if total_workers != -1:
        for t in MILESTONES:
            key = f"Workers{t}"
            if data.get(key, -1) == -1:
                data[key] = total_workers
    else:
        for t in MILESTONES:
            key = f"Workers{t}"
            if data.get(key, -1) == -1:
                data[key] = 0

    # Combat units: Light / Heavy / Ranged
    for unit in ("Light", "Heavy", "Ranged"):
        last = None
        for t in MILESTONES:
            key = f"{unit}{t}"
            v = data.get(key, -1)
            if v != -1:
                last = v
            else:
                if last is not None:
                    data[key] = last
        total_key = f"Total{unit}"
        total_val = data.get(total_key, -1)
        if total_val != -1:
            for t in MILESTONES:
                key = f"{unit}{t}"
                if data.get(key, -1) == -1:
                    data[key] = total_val
        else:
            for t in MILESTONES:
                key = f"{unit}{t}"
                if data.get(key, -1) == -1:
                    data[key] = 0

    # AvgDist at milestones: carry forward last seen distance
    last = None
    for t in MILESTONES:
        key = f"AvgDist{t}"
        v = data.get(key, -1)
        if v != -1:
            last = v
        else:
            if last is not None:
                data[key] = last
    # If still missing at all milestones, set to 0
    for t in MILESTONES:
        key = f"AvgDist{t}"
        if data.get(key, -1) == -1:
            data[key] = 0.0


def parse_block(block: str) -> dict:
    first_line = block.splitlines()[0].strip()
    m_pid = re.search(r"player\s+(\d+)\s+vs\s+(\d+)", first_line)
    if m_pid:
        player_id = int(m_pid.group(1))
    else:
        player_id = 0

    data = {k: -1 for k in COLS}
    # default: if no barracks is ever seen, keep FirstBarracks = 0
    data["FirstBarracks"] = 0

    lines = block.splitlines()
    for line in lines:
        line = line.strip()

        m = re.search(rf"Player {player_id} first has a Barracks at trace time = (\d+)", line)
        if m:
            data["FirstBarracks"] = int(m.group(1))

        m = re.search(rf"Player {player_id} built (\d+) Barracks", line)
        if m:
            data["NumBarracks"] = int(m.group(1))

        m = re.search(r"time >= (\d+) \(first time=\d+\): Light=(\d+), Heavy=(\d+), Ranged=(\d+)", line)
        if m:
            t, L, H, R = map(int, m.groups())
            if t in (100, 200, 400, 800):
                data[f"Light{t}"] = L
                data[f"Heavy{t}"] = H
                data[f"Ranged{t}"] = R

        m = re.search(r"time >= (\d+) \(first time=\d+\): avg distance = ([0-9.]+)", line)
        if m:
            t = int(m.group(1))
            if t in (100, 200, 400, 800):
                data[f"AvgDist{t}"] = float(m.group(2))

        m = re.search(rf"Max ATTACKs by player {player_id} in a single trace entry: (\d+)", line)
        if m:
            data["MaxAttacksSingleTrace"] = int(m.group(1))

        m = re.search(r"Occurred at trace time = (\d+)", line)
        if m:
            data["TimeMaxAttacks"] = int(m.group(1))

        m = re.search(rf"Player {player_id} issued (\d+) ATTACK actions in total", line)
        if m:
            data["TotalAttacks"] = int(m.group(1))

        m = re.search(r"Average ATTACKs per trace entry: ([0-9.]+)", line)
        if m:
            data["AvgAttacksPerTrace"] = float(m.group(1))

        m = re.search(rf"Player {player_id} first ATTACK action at trace time = (\d+)", line)
        if m:
            data["FirstAttackTime"] = int(m.group(1))

        m = re.search(r"Light units\s*:\s*(\d+)", line)
        if m:
            data["TotalLight"] = int(m.group(1))
        m = re.search(r"Heavy units\s*:\s*(\d+)", line)
        if m:
            data["TotalHeavy"] = int(m.group(1))
        m = re.search(r"Ranged units\s*:\s*(\d+)", line)
        if m:
            data["TotalRanged"] = int(m.group(1))

        m = re.search(r"Game length:\s*(\d+)\s*ticks", line)
        if m:
            data["GameLength"] = int(m.group(1))

        if f"Player {player_id}: WIN" in line:
            data["PlayerWin"] = 1
        elif f"Player {player_id}: LOSS" in line:
            data["PlayerWin"] = 0
        elif f"Player {player_id}: DRAW" in line:
            data["PlayerWin"] = 0.5

        m = re.search(r"time >= (\d+): (\d+) Workers", line)
        if m:
            t = int(m.group(1))
            if t in (100, 200, 400, 800):
                data[f"Workers{t}"] = int(m.group(2))

        m = re.search(rf"Player {player_id} built (\d+) Workers in this trace", line)
        if m:
            data["TotalWorkers"] = int(m.group(1))

        m = re.search(rf"Player {player_id} harvested ~([0-9.]+)% of all resources", line)
        if m:
            data["PercentMapHarvested"] = float(m.group(1))

    # fix milestones so no -1 remains in workers / combat units / avg distances
    carry_forward_milestones(data)

    # also remove -1 from other numeric fields (turn into 0)
    for k in COLS:
        if isinstance(data[k], (int, float)) and data[k] == -1:
            data[k] = 0

    return data

rows = [parse_block(b) for b in blocks]

print(",".join(COLS))
for d in rows:
    print(",".join(str(d[k]) for k in COLS))


FirstBarracks,NumBarracks,Light100,Heavy100,Ranged100,Light200,Heavy200,Ranged200,Light400,Heavy400,Ranged400,Light800,Heavy800,Ranged800,AvgDist100,AvgDist200,AvgDist400,AvgDist800,MaxAttacksSingleTrace,TimeMaxAttacks,TotalAttacks,AvgAttacksPerTrace,FirstAttackTime,TotalLight,TotalHeavy,TotalRanged,GameLength,PlayerWin,Workers100,Workers200,Workers400,Workers800,TotalWorkers,PercentMapHarvested
0,0,0,0,0,0,0,0,0,0,0,0,0,0,11.068,8.399,8.399,8.399,1,230,12,0.0504,230,0,0,0,305,1,3,5,5,5,7,26.25
0,0,0,0,0,0,0,0,0,0,0,0,0,0,11.068,8.399,8.399,8.399,1,230,12,0.0504,230,0,0,0,305,1,3,5,5,5,7,26.25
0,0,0,0,0,0,0,0,0,0,0,0,0,0,9.499,8.046,8.046,8.046,1,190,12,0.0519,190,0,0,0,315,1,3,5,5,5,7,26.25
0,0,0,0,0,0,0,0,0,0,0,0,0,0,8.106,6.685,6.685,6.685,1,170,12,0.0472,170,0,0,0,325,1,3,5,5,5,7,26.25
0,0,0,0,0,0,0,0,0,0,0,0,0,0,6.978,5.865,5.865,5.865,1,170,13,0.0695,170,0,0,0,335,1,3,5,5,5,7,27.5
0,0,0,0,0,0,0,0,0,0,0,0,0,0,6.24,6.24,6.24,6.24,1,130,12,0.0566,130,0,0,0,335,1,3,5,5,5,7,27.5
0,0,0

In [3]:
import pandas as pd
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

%matplotlib inline

df_raw = pd.read_csv("clusteringTrace.csv", sep=";", engine="python", header=0)

# Clean
df = df_raw.replace("None", pd.NA)
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].str.replace(",", ".", regex=False)

df_numeric = df.apply(pd.to_numeric, errors="coerce")
df_numeric = df_numeric.dropna(axis=1, how="all")   # drop non‑numeric columns

print("Numeric columns:", list(df_numeric.columns))

# Keep only rows that have at least one numeric value (but NOT drop them from df_raw)
valid_rows = df_numeric.dropna(axis=0, how="all")
print("Rows used for clustering:", len(valid_rows))

# Fill NaNs in those rows
valid_filled = valid_rows.fillna(valid_rows.mean(numeric_only=True))

# ----- SCALE THEN WEIGHT FEATURES -----
scaler = StandardScaler()
X_scaled = scaler.fit_transform(valid_filled)

# Put back into a DataFrame so we can edit specific columns
X_df = pd.DataFrame(X_scaled, columns=valid_filled.columns, index=valid_filled.index)

# Downweight FirstBarracks and NumBarracks by factor 0.1
for col in ["FirstBarracks", "NumBarracks"]:
    if col in X_df.columns:
        X_df[col] = X_df[col] * 0.0001

X_weighted = X_df.values

# K-means on weighted features
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(X_weighted)

# Create a new column in df_raw, default -1, then fill only for valid_rows
df_raw["cluster"] = -1
df_raw.loc[valid_filled.index, "cluster"] = cluster_labels

# Show head
display(df_raw.head())

# Cluster centers in scaled+weighted space
centers = pd.DataFrame(kmeans.cluster_centers_, columns=valid_filled.columns)
print("\nCluster centers (scaled & weighted features):")
display(centers)

# Plot (use same valid rows) - pick any two features that exist
x_feature = "PercentMapHarvested"
y_feature = "AvgAttacksPerTrace"

if x_feature in valid_filled.columns and y_feature in valid_filled.columns:
    plt.figure(figsize=(7, 6))
    sns.scatterplot(
        x=valid_filled[x_feature],
        y=valid_filled[y_feature],
        hue=cluster_labels,
        palette="tab10"
    )
    plt.xlabel(x_feature)
    plt.ylabel(y_feature)
    plt.title(f"K-Means clusters (k={k})")
    plt.legend(title="Cluster")
    plt.tight_layout()
    plt.show()
else:
    print(f"Cannot plot: {x_feature} or {y_feature} not in numeric columns.")

# Make a copy that includes cluster labels, but only for valid rows
clustered = df_raw.loc[valid_filled.index].copy()
clustered["cluster"] = cluster_labels

# 1) Quick view: first few rows with clusters
display(clustered.head())

# 2) Show indices belonging to each cluster
for c in sorted(clustered["cluster"].unique()):
    idx = clustered.index[clustered["cluster"] == c].tolist()
    print(f"\nCluster {c}: row indices {idx}")

# 3) Example: show full rows for cluster 0
cluster_id = 0
display(clustered[clustered["cluster"] == cluster_id])


Numeric columns: ['FirstBarracks            : -1', 'NumBarracks              : 0', 'Light100                 : 0', 'Heavy100                 : 0', 'Ranged100                : 0', 'Light200                 : 0', 'Heavy200                 : 0', 'Ranged200                : 0', 'Light400                 : None', 'Heavy400                 : None', 'Ranged400                : None', 'Light800                 : None', 'Heavy800                 : None', 'Ranged800                : None', 'AvgDist100               : 6.096', 'AvgDist200               : 5.7', 'AvgDist400               : None', 'AvgDist800               : None', 'MaxAttacksSingleTrace    : 1', 'TimeMaxAttacks           : 180', 'TotalAttacks             : 15', 'AvgAttacksPerTrace       : 0.2679', 'FirstAttackTime          : 180', 'TotalLight               : 0', 'TotalHeavy               : 0', 'TotalRanged              : 0', 'GameLength               : 345', 'Player0Win               : 1', 'Workers100               : 2', 'Workers200

,FirstBarracks : -1,NumBarracks : 0,Light100 : 0,Heavy100 : 0,Ranged100 : 0,Light200 : 0,Heavy200 : 0,Ranged200 : 0,Light400 : None,Heavy400 : None,...,TotalRanged : 0,GameLength : 345,Player0Win : 1,Workers100 : 2,Workers200 : 4,Workers400 : None,Workers800 : None,TotalWorkers,PercentMapHarvested,cluster
0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,345.0,1.0,2.0,4.0,-1.0,-1.0,6.0,47.5,0
1,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,375.0,1.0,2.0,4.0,-1.0,-1.0,7.0,52.5,0
2,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,385.0,1.0,2.0,3.0,-1.0,-1.0,7.0,52.5,0
3,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,516.0,1.0,2.0,4.0,7.0,-1.0,10.0,62.5,1
4,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,375.0,1.0,2.0,4.0,-1.0,-1.0,7.0,52.5,0



Cluster centers (scaled & weighted features):


,FirstBarracks : -1,NumBarracks : 0,Light100 : 0,Heavy100 : 0,Ranged100 : 0,Light200 : 0,Heavy200 : 0,Ranged200 : 0,Light400 : None,Heavy400 : None,...,TotalHeavy : 0,TotalRanged : 0,GameLength : 345,Player0Win : 1,Workers100 : 2,Workers200 : 4,Workers400 : None,Workers800 : None,TotalWorkers,PercentMapHarvested
0,-0.264868,-0.267261,0.0,0.0,0.0,0.0,0.0,0.0,-0.654654,-0.654654,...,0.0,0.0,-0.514155,0.0,0.0,-0.079584,-0.654070,0.0,-0.523084,-0.417070
1,-0.264868,-0.267261,0.0,0.0,0.0,0.0,0.0,0.0,1.527525,1.527525,...,0.0,0.0,0.962468,0.0,0.0,0.185695,1.518377,0.0,1.010565,0.818693
2,3.708158,3.741657,0.0,0.0,0.0,0.0,0.0,0.0,1.527525,1.527525,...,0.0,0.0,2.029988,0.0,0.0,0.185695,1.553416,0.0,1.955401,1.513811


Cannot plot: PercentMapHarvested or AvgAttacksPerTrace not in numeric columns.


,FirstBarracks : -1,NumBarracks : 0,Light100 : 0,Heavy100 : 0,Ranged100 : 0,Light200 : 0,Heavy200 : 0,Ranged200 : 0,Light400 : None,Heavy400 : None,...,TotalRanged : 0,GameLength : 345,Player0Win : 1,Workers100 : 2,Workers200 : 4,Workers400 : None,Workers800 : None,TotalWorkers,PercentMapHarvested,cluster
0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,345.0,1.0,2.0,4.0,-1.0,-1.0,6.0,47.5,0
1,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,375.0,1.0,2.0,4.0,-1.0,-1.0,7.0,52.5,0
2,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,385.0,1.0,2.0,3.0,-1.0,-1.0,7.0,52.5,0
3,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,516.0,1.0,2.0,4.0,7.0,-1.0,10.0,62.5,1
4,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,375.0,1.0,2.0,4.0,-1.0,-1.0,7.0,52.5,0



Cluster 0: row indices [0, 1, 2, 4, 5, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 23, 26, 27, 28, 29]

Cluster 1: row indices [3, 6, 7, 8, 9, 22, 24]

Cluster 2: row indices [21, 25]


,FirstBarracks : -1,NumBarracks : 0,Light100 : 0,Heavy100 : 0,Ranged100 : 0,Light200 : 0,Heavy200 : 0,Ranged200 : 0,Light400 : None,Heavy400 : None,...,TotalRanged : 0,GameLength : 345,Player0Win : 1,Workers100 : 2,Workers200 : 4,Workers400 : None,Workers800 : None,TotalWorkers,PercentMapHarvested,cluster
0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,345.0,1.0,2.0,4.0,-1.0,-1.0,6.0,47.5,0
1,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,375.0,1.0,2.0,4.0,-1.0,-1.0,7.0,52.5,0
2,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,385.0,1.0,2.0,3.0,-1.0,-1.0,7.0,52.5,0
4,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,375.0,1.0,2.0,4.0,-1.0,-1.0,7.0,52.5,0
5,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,390.0,1.0,2.0,4.0,-1.0,-1.0,7.0,50.0,0
10,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,335.0,1.0,2.0,4.0,-1.0,-1.0,6.0,37.5,0
11,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,295.0,1.0,2.0,4.0,-1.0,-1.0,5.0,35.0,0
12,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,275.0,1.0,2.0,4.0,-1.0,-1.0,5.0,35.0,0
13,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,335.0,1.0,2.0,4.0,-1.0,-1.0,6.0,37.5,0
14,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,315.0,1.0,2.0,4.0,-1.0,-1.0,6.0,37.5,0


In [2]:
import pandas as pd
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

%matplotlib inline

df_raw = pd.read_csv("clusteringTrace.csv", sep=";", engine="python", header=0)

# Clean
df = df_raw.replace("None", pd.NA)
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].str.replace(",", ".", regex=False)

df_numeric = df.apply(pd.to_numeric, errors="coerce")
df_numeric = df_numeric.dropna(axis=1, how="all")   # drop non‑numeric columns

print("Numeric columns (before drop):", list(df_numeric.columns))

# Keep only rows that have at least one numeric value
valid_rows = df_numeric.dropna(axis=0, how="all")
print("Rows used for clustering:", len(valid_rows))

# Fill NaNs in those rows
valid_filled = valid_rows.fillna(valid_rows.mean(numeric_only=True))

# ---- DROP UNWANTED COLUMNS FOR CLUSTERING ----
cols_to_drop = [
    "Light100", "Light200", "Light400",
    "Heavy100", "Heavy200", "Heavy400",
    "Ranged100", "Ranged200", "Ranged400",
    "Player0Win", "FirstBarracks", "NumBarracks",
]

existing_drop = [c for c in cols_to_drop if c in valid_filled.columns]
print("Dropping from clustering:", existing_drop)

valid_filled = valid_filled.drop(columns=existing_drop)

print("Numeric columns (used for clustering):", list(valid_filled.columns))

# ----- SCALE FEATURES -----
scaler = StandardScaler()
X_scaled = scaler.fit_transform(valid_filled)

X_df = pd.DataFrame(X_scaled, columns=valid_filled.columns, index=valid_filled.index)

# (no need to downweight FirstBarracks/NumBarracks now, they are removed)

X_weighted = X_df.values

# K-means on selected features
k = 3
kmeans = KMeans(n_clusters=k, random_state=0)
cluster_labels = kmeans.fit_predict(X_weighted)

# Create a new column in df_raw, default -1, then fill only for valid_rows
df_raw["cluster"] = -1
df_raw.loc[valid_filled.index, "cluster"] = cluster_labels

# Show head
display(df_raw.head())

# Cluster centers in scaled feature space
centers = pd.DataFrame(kmeans.cluster_centers_, columns=valid_filled.columns)
print("\nCluster centers (scaled features):")
display(centers)

# Plot (use same valid rows) - pick any two features that exist
x_feature = "PercentMapHarvested"
y_feature = "AvgAttacksPerTrace"

if x_feature in valid_filled.columns and y_feature in valid_filled.columns:
    plt.figure(figsize=(7, 6))
    sns.scatterplot(
        x=valid_filled[x_feature],
        y=valid_filled[y_feature],
        hue=cluster_labels,
        palette="tab10"
    )
    plt.xlabel(x_feature)
    plt.ylabel(y_feature)
    plt.title(f"K-Means clusters (k={k})")
    plt.legend(title="Cluster")
    plt.tight_layout()
    plt.show()
else:
    print(f"Cannot plot: {x_feature} or {y_feature} not in numeric columns.")
    print("Available:", list(valid_filled.columns))

# Make a copy that includes cluster labels, but only for valid rows
clustered = df_raw.loc[valid_filled.index].copy()
clustered["cluster"] = cluster_labels

# 1) Quick view: first few rows with clusters
display(clustered.head())

# 2) Show indices belonging to each cluster
for c in sorted(clustered["cluster"].unique()):
    idx = clustered.index[clustered["cluster"] == c].tolist()
    print(f"\nCluster {c}: row indices {idx}")

# 3) Example: show full rows for cluster 0
cluster_id = 0
display(clustered[clustered["cluster"] == cluster_id])


Numeric columns (before drop): ['FirstBarracks            : -1', 'NumBarracks              : 0', 'Light100                 : 0', 'Heavy100                 : 0', 'Ranged100                : 0', 'Light200                 : 0', 'Heavy200                 : 0', 'Ranged200                : 0', 'Light400                 : None', 'Heavy400                 : None', 'Ranged400                : None', 'Light800                 : None', 'Heavy800                 : None', 'Ranged800                : None', 'AvgDist100               : 6.096', 'AvgDist200               : 5.7', 'AvgDist400               : None', 'AvgDist800               : None', 'MaxAttacksSingleTrace    : 1', 'TimeMaxAttacks           : 180', 'TotalAttacks             : 15', 'AvgAttacksPerTrace       : 0.2679', 'FirstAttackTime          : 180', 'TotalLight               : 0', 'TotalHeavy               : 0', 'TotalRanged              : 0', 'GameLength               : 345', 'Player0Win               : 1', 'Workers100               : 2

,FirstBarracks : -1,NumBarracks : 0,Light100 : 0,Heavy100 : 0,Ranged100 : 0,Light200 : 0,Heavy200 : 0,Ranged200 : 0,Light400 : None,Heavy400 : None,...,TotalRanged : 0,GameLength : 345,Player0Win : 1,Workers100 : 2,Workers200 : 4,Workers400 : None,Workers800 : None,TotalWorkers,PercentMapHarvested,cluster
0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,345.0,1.0,2.0,4.0,-1.0,-1.0,6.0,47.5,0
1,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,375.0,1.0,2.0,4.0,-1.0,-1.0,7.0,52.5,0
2,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,385.0,1.0,2.0,3.0,-1.0,-1.0,7.0,52.5,0
3,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,516.0,1.0,2.0,4.0,7.0,-1.0,10.0,62.5,1
4,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,375.0,1.0,2.0,4.0,-1.0,-1.0,7.0,52.5,0



Cluster centers (scaled features):


,FirstBarracks : -1,NumBarracks : 0,Light100 : 0,Heavy100 : 0,Ranged100 : 0,Light200 : 0,Heavy200 : 0,Ranged200 : 0,Light400 : None,Heavy400 : None,...,TotalHeavy : 0,TotalRanged : 0,GameLength : 345,Player0Win : 1,Workers100 : 2,Workers200 : 4,Workers400 : None,Workers800 : None,TotalWorkers,PercentMapHarvested
0,-0.264868,-0.267261,0.0,0.0,0.0,0.0,0.0,0.0,-0.654654,-0.654654,...,0.0,0.0,-0.514155,0.0,0.0,-0.079584,-0.654070,0.0,-0.523084,-0.417070
1,-0.264868,-0.267261,0.0,0.0,0.0,0.0,0.0,0.0,1.527525,1.527525,...,0.0,0.0,0.962468,0.0,0.0,0.185695,1.518377,0.0,1.010565,0.818693
2,3.708158,3.741657,0.0,0.0,0.0,0.0,0.0,0.0,1.527525,1.527525,...,0.0,0.0,2.029988,0.0,0.0,0.185695,1.553416,0.0,1.955401,1.513811


Cannot plot: PercentMapHarvested or AvgAttacksPerTrace not in numeric columns.
Available: ['FirstBarracks            : -1', 'NumBarracks              : 0', 'Light100                 : 0', 'Heavy100                 : 0', 'Ranged100                : 0', 'Light200                 : 0', 'Heavy200                 : 0', 'Ranged200                : 0', 'Light400                 : None', 'Heavy400                 : None', 'Ranged400                : None', 'Light800                 : None', 'Heavy800                 : None', 'Ranged800                : None', 'AvgDist100               : 6.096', 'AvgDist200               : 5.7', 'AvgDist400               : None', 'AvgDist800               : None', 'MaxAttacksSingleTrace    : 1', 'TimeMaxAttacks           : 180', 'TotalAttacks             : 15', 'AvgAttacksPerTrace       : 0.2679', 'FirstAttackTime          : 180', 'TotalLight               : 0', 'TotalHeavy               : 0', 'TotalRanged              : 0', 'GameLength               : 345', 'P

,FirstBarracks : -1,NumBarracks : 0,Light100 : 0,Heavy100 : 0,Ranged100 : 0,Light200 : 0,Heavy200 : 0,Ranged200 : 0,Light400 : None,Heavy400 : None,...,TotalRanged : 0,GameLength : 345,Player0Win : 1,Workers100 : 2,Workers200 : 4,Workers400 : None,Workers800 : None,TotalWorkers,PercentMapHarvested,cluster
0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,345.0,1.0,2.0,4.0,-1.0,-1.0,6.0,47.5,0
1,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,375.0,1.0,2.0,4.0,-1.0,-1.0,7.0,52.5,0
2,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,385.0,1.0,2.0,3.0,-1.0,-1.0,7.0,52.5,0
3,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,516.0,1.0,2.0,4.0,7.0,-1.0,10.0,62.5,1
4,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,375.0,1.0,2.0,4.0,-1.0,-1.0,7.0,52.5,0



Cluster 0: row indices [0, 1, 2, 4, 5, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 23, 26, 27, 28, 29]

Cluster 1: row indices [3, 6, 7, 8, 9, 22, 24]

Cluster 2: row indices [21, 25]


,FirstBarracks : -1,NumBarracks : 0,Light100 : 0,Heavy100 : 0,Ranged100 : 0,Light200 : 0,Heavy200 : 0,Ranged200 : 0,Light400 : None,Heavy400 : None,...,TotalRanged : 0,GameLength : 345,Player0Win : 1,Workers100 : 2,Workers200 : 4,Workers400 : None,Workers800 : None,TotalWorkers,PercentMapHarvested,cluster
0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,345.0,1.0,2.0,4.0,-1.0,-1.0,6.0,47.5,0
1,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,375.0,1.0,2.0,4.0,-1.0,-1.0,7.0,52.5,0
2,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,385.0,1.0,2.0,3.0,-1.0,-1.0,7.0,52.5,0
4,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,375.0,1.0,2.0,4.0,-1.0,-1.0,7.0,52.5,0
5,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,390.0,1.0,2.0,4.0,-1.0,-1.0,7.0,50.0,0
10,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,335.0,1.0,2.0,4.0,-1.0,-1.0,6.0,37.5,0
11,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,295.0,1.0,2.0,4.0,-1.0,-1.0,5.0,35.0,0
12,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,275.0,1.0,2.0,4.0,-1.0,-1.0,5.0,35.0,0
13,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,335.0,1.0,2.0,4.0,-1.0,-1.0,6.0,37.5,0
14,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,315.0,1.0,2.0,4.0,-1.0,-1.0,6.0,37.5,0
